# KneeMIL: training outside Kaggle (Colab or a local NVIDIA GPU)

Trains one run on your own GPU and uploads the checkpoints as a **private Kaggle dataset**.
The Kaggle submit kernel then uses them, and a submit only costs ~15 min of Kaggle GPU quota.

**Before you start (one time):**
1. **Colab:** *Runtime → Change runtime type → T4 GPU* (or better).
2. **Colab:** open the 🔑 *Secrets* panel and add:
   * `KAGGLE_API_TOKEN`: a Kaggle API token (kaggle.com → Settings → API).
   * `KAGGLE_USERNAME`: `danielaijdkx`.

   Never paste the token into a cell. On a local PC, export both as environment variables instead.
3. Checkpoints are written to Google Drive (`MyDrive/kneemri/<run>`). If the session disconnects,
   re-run all cells: finished folds are skipped.

Run the cells top to bottom.

In [ ]:
# ---- run configuration (r7 = recentred 120 mm crop, labels refined by the r1+r2+r3+r5+r6 OOF ensemble)
RUN_NAME    = "r7"
CACHES      = ["kneemri-cache120a", "kneemri-cache120b"]      # Kaggle kernels holding the .npz cache
LABELS      = "kneemri-labels"                                   # teacher table kernel
REFINE_FROM = ["kneemri-train-r1", "kneemri-train-r2", "kneemri-train-r3",
               "kneemri-train-r5", "kneemri-train-r6"]            # their OOF fill the silent label cells
BACKBONE    = "convnext_tiny.fb_in22k_ft_in1k_384"
N_FOLDS, EPOCHS, MAX_WINDOWS = 5, 10, 16
BATCH, ACCUM = 4, 2          # effective batch 8; T4-safe for ConvNeXt-T at 320 px
BRANCH      = "claude/nifty-cannon-ivkdzl"
KAGGLE_USER = "danielaijdkx"
DATASET     = f"{KAGGLE_USER}/kneemri-{RUN_NAME}"               # private dataset the checkpoints go to

In [ ]:
import os, sys, subprocess, json, time, shutil, glob
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    for k in ("KAGGLE_API_TOKEN", "KAGGLE_USERNAME"):
        os.environ[k] = userdata.get(k)
    ROOT, OUT_ROOT = Path("/content"), Path("/content/drive/MyDrive/kneemri")
else:
    assert os.environ.get("KAGGLE_API_TOKEN"), "export KAGGLE_API_TOKEN and KAGGLE_USERNAME first"
    ROOT = OUT_ROOT = Path.home() / "kneemri_external"
WORK, CODE = ROOT / "work", ROOT / "RSNA"
OUT = OUT_ROOT / RUN_NAME
for p in (WORK, OUT):
    p.mkdir(parents=True, exist_ok=True)

def sh(cmd, check=True):
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True)
    if check and r.returncode:
        raise RuntimeError(f"failed ({r.returncode}): {cmd}")
    return r.returncode

sh("nvidia-smi --query-gpu=name,memory.total --format=csv")
sh("pip install -q -U kaggle timm pydicom pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg")
if not CODE.exists():
    sh(f"git clone -q -b {BRANCH} https://github.com/Daniel766hi/RSNA {CODE}")
else:
    sh(f"git -C {CODE} pull -q")
sh(f"pip install -q -e {CODE}")

In [ ]:
# ---- data: train.csv, the cache shards, the teacher labels, and the OOF of the refining runs
DATA = WORK / "data"
DATA.mkdir(exist_ok=True)
if not (DATA / "train.csv").exists():
    sh(f"kaggle competitions download rsna-knee-abnormality-detection -f train.csv -p {DATA}")
    for z in DATA.glob("*.zip"):
        sh(f"cd {DATA} && unzip -o -q {z.name} && rm {z.name}")

def kernel_output(slug, pattern, dest):
    dest = WORK / dest
    if dest.exists() and any(dest.rglob("*")):
        print("cached:", dest); return dest
    dest.mkdir(parents=True, exist_ok=True)
    sh(f"kaggle kernels output {KAGGLE_USER}/{slug} -p {dest} --file-pattern '{pattern}'")
    return dest

cache_dirs = []
for c in CACHES:                                   # ~10 GB per shard
    d = kernel_output(c, r".*\.npz$|volume\.json$", c)
    cache_dirs.append(next(p.parent for p in d.rglob("*.npz")))
labels = next(kernel_output(LABELS, r"labels\.csv$", LABELS).rglob("labels.csv"))
oof_globs = []
for r in REFINE_FROM:
    d = kernel_output(r, r"oof_fold.*\.csv$", r)
    oof_globs.append(str(next(d.rglob("oof_fold0.csv")).parent / "oof_fold*.csv"))
print("caches:", cache_dirs, "\nlabels:", labels, "\noof:", oof_globs)

In [ ]:
# ---- labels: silent report cells filled by the rank-averaged OOF of the refining runs (lever L1)
refined = WORK / f"labels_{RUN_NAME}.csv"
if not refined.exists():
    sh(f"python {CODE}/scripts/refine_labels.py --data {DATA} --labels {labels} "
       + " ".join(f"--oof '{g}'" for g in oof_globs) + f" --out {refined}")
CACHE_ARG = ",".join(map(str, cache_dirs))
COMMON = (f"python {CODE}/scripts/train_cv.py --data {DATA} --cache {CACHE_ARG} --labels {refined} "
          f"--backbone {BACKBONE} --batch-size {BATCH} --grad-accum {ACCUM} --n-folds {N_FOLDS} "
          f"--max-windows {MAX_WINDOWS} --workers {max(1, (os.cpu_count() or 2) - 1)}")

In [ ]:
# ---- timing probe: 1 epoch of fold 0 on ~300 studies, extrapolated to the full run
probe = WORK / "probe"
t0 = time.time()
sh(f"{COMMON} --out {probe} --folds 0 --epochs 1 --limit 300")
sec = time.time() - t0
n_train_full = 4407 * (N_FOLDS - 1) / N_FOLDS
n_train_probe = (300 + 58) * (N_FOLDS - 1) / N_FOLDS
per_fold_h = sec * (n_train_full / n_train_probe) * EPOCHS / 3600
print(f"probe {sec/60:.1f} min -> ~{per_fold_h:.1f} h per fold, ~{per_fold_h * N_FOLDS:.1f} h for {N_FOLDS} folds")
print("Colab sessions can end after a few hours; finished folds are kept on Drive, so just re-run.")

In [ ]:
# ---- train fold by fold (resumable: folds with a checkpoint on Drive are skipped)
for k in range(N_FOLDS):
    if (OUT / f"fold{k}.pt").exists() and (OUT / f"oof_fold{k}.csv").exists():
        print(f"fold {k}: done, skipped"); continue
    t0 = time.time()
    sh(f"{COMMON} --out {OUT} --folds {k} --epochs {EPOCHS}")
    print(f"fold {k}: {(time.time() - t0) / 3600:.2f} h", flush=True)
vol = next(Path(cache_dirs[0]).parent.rglob("volume.json"), None)
if vol:   # the submit stage reads this to decode test studies with the same recipe
    shutil.copy(vol, OUT / "volume.json")
for m in sorted(OUT.glob("metrics*.json")):
    print(m.name, json.loads(m.read_text()).get("gold_macro"))

In [ ]:
# ---- upload the run as a private Kaggle dataset (new version if it exists)
done = sorted(OUT.glob("fold*.pt"))
assert len(done) == N_FOLDS, f"only {len(done)}/{N_FOLDS} folds trained; re-run the training cell"
(OUT / "dataset-metadata.json").write_text(json.dumps(
    {"title": f"kneemri-{RUN_NAME}", "id": DATASET, "licenses": [{"name": "CC0-1.0"}]}, indent=1))
exists = subprocess.run(f"kaggle datasets status {DATASET}", shell=True, capture_output=True, text=True)
if exists.returncode == 0 and "ready" in exists.stdout.lower():
    sh(f"kaggle datasets version -p {OUT} -m 'update {RUN_NAME}'")
else:
    sh(f"kaggle datasets create -p {OUT}")
print(f"uploaded -> https://www.kaggle.com/datasets/{DATASET}")
print(f"Next: open colab/submit.ipynb and set TRAIN_DATASETS = [\"{DATASET}\"], then Run all.")